In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Sales Dataframe

sales_data = [
    (1, 501, 301, 101, '2025-05-11', 2, 550.0),  
    (2, 502, 302, 102, '2025-05-12', 1, 999.0),   
    (3, 503, 303, 103, '2025-05-13', 3, 250.0),   
    (4, 504, 304, 104, '2025-05-14', 0, 450.0),   
    (5, 505, 305, 105, '2025-05-15', 1, 600.0), 
    (6, 506, None, 106, '2025-05-16', 2, 300.0),  
    (7, 507, 307, None, '2025-05-17', 4, 750.0), 
    (8, 508, 301, 108, '2025-05-18', 1, 1200.0), 
    (9, 509, 309, 109, '2025-05-19', 5, 180.0),  
    (10, 510, 310, 110, '2025-05-20', 2, 850.0),
    (11, None, 302, 111, '2025-05-21', 3, 950.0), 
    (12, 511, 312, 112, '2025-05-22', 1, 0.0),    
    (13, 512, 313, 113, '2025-05-23', 6, 220.0),
    (14, 501, 304, 114, '2025-05-24', 2, 650.0),  
    (15, 513, 315, 115, '2025-05-25', 1, 400.0)
]

sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("product_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("store_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True)
])

sales_df = spark.createDataFrame(data=sales_data, schema=sales_schema)
sales_df = sales_df.withColumn("sale_date", to_date(col("sale_date"), "yyyy-MM-dd"))
sales_df.printSchema()
sales_df.display()


root
 |-- sale_id: integer (nullable = false)
 |-- product_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)



sale_id,product_id,customer_id,store_id,sale_date,quantity,unit_price
1,501,301,101,2025-05-11,2,550.0
2,502,302,102,2025-05-12,1,999.0
3,503,303,103,2025-05-13,3,250.0
4,504,304,104,2025-05-14,0,450.0
5,505,305,105,2025-05-15,1,600.0
6,506,null,106,2025-05-16,2,300.0
7,507,307,null,2025-05-17,4,750.0
8,508,301,108,2025-05-18,1,1200.0
9,509,309,109,2025-05-19,5,180.0
10,510,310,110,2025-05-20,2,850.0


In [0]:
#product dataframe

products_data = [
    (501, "LED TV", "Electronics", 801),
    (502, "Laptop", "Electronics", 802),
    (503, "Mixer Grinder", "Home Appliance", 803),
    (504, "Iron Box", "Home Appliance", 804),
    (505, "Water Purifier", "Home Appliance", 805),
    (506, "Ceiling Fan", "Electricals", 806),
    (507, "Washing Machine", "Appliances", 807),
    (508, "Smartphone", "Mobiles", 808),
    (509, "Bluetooth Speaker", "Electronics", 809),
    (510, "Air Conditioner", "Appliances", 810),
    (511, "Electric Kettle", "Kitchen", 811),
    (512, "Induction Stove", "Kitchen", 812),
    (513, "Microwave Oven", "Kitchen", 813)
]

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("supplier_id", IntegerType(), True)
])

products_df = spark.createDataFrame(products_data, schema=products_schema)
products_df.printSchema()
products_df.display()


root
 |-- product_id: integer (nullable = false)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- supplier_id: integer (nullable = true)



product_id,product_name,category,supplier_id
501,LED TV,Electronics,801
502,Laptop,Electronics,802
503,Mixer Grinder,Home Appliance,803
504,Iron Box,Home Appliance,804
505,Water Purifier,Home Appliance,805
506,Ceiling Fan,Electricals,806
507,Washing Machine,Appliances,807
508,Smartphone,Mobiles,808
509,Bluetooth Speaker,Electronics,809
510,Air Conditioner,Appliances,810


In [0]:
#customers dataframe

customers_data = [
    (301, "Nitheesh", "North", "2023-01-10"),
    (302, "Raju", "West", "2023-02-12"),
    (303, "Chandu", "South", "2023-03-01"),
    (304, "Sai", "East", "2023-04-05"),
    (305, "Charan", "North", "2023-05-10"),
    (306, "Satish", "West", "2023-06-14"),
    (307, "Narendra", "South", "2023-07-09"),
    (308, "Naveen", "East", "2023-08-15"),
    (309, "Vamsi", "North", "2023-09-02"),
    (310, "Sanjay", "West", "2023-09-20"),
    (311, "Manikanta", "South", "2023-10-11"),
    (312, "Ajay", "East", "2023-10-28"),
    (313, "Vikram", "North", "2023-11-18"),
    (314, "Rahul", "West", "2023-11-28"),
    (315, "Ravi", "South", "2023-12-05")
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("customer_name", StringType(), True),
    StructField("region", StringType(), True),
    StructField("join_date", StringType(), True)
])

customer_columns = ["customer_id", "customer_name", "region", "join_date"]

customers_df = spark.createDataFrame(customers_data, schema=customers_schema)
customers_df = customers_df.withColumn("join_date", to_date(col("join_date"), "yyyy-MM-dd"))
customers_df.printSchema()
customers_df.display()


root
 |-- customer_id: integer (nullable = false)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- join_date: date (nullable = true)



customer_id,customer_name,region,join_date
301,Nitheesh,North,2023-01-10
302,Raju,West,2023-02-12
303,Chandu,South,2023-03-01
304,Sai,East,2023-04-05
305,Charan,North,2023-05-10
306,Satish,West,2023-06-14
307,Narendra,South,2023-07-09
308,Naveen,East,2023-08-15
309,Vamsi,North,2023-09-02
310,Sanjay,West,2023-09-20


In [0]:
#stores dataframe
stores_data = [
    (101, "Reliance Digital", "North"),
    (102, "Croma", "West"),
    (103, "Vijay Sales", "South"),
    (104, "Ezone", "East"),
    (105, "Spencers", "North"),
    (106, "Big Bazaar", "West"),
    (107, "More Retail", "South"),
    (108, "D Mart", "East"),
    (109, "Metro", "North"),
    (110, "Best Price", "West"),
    (111, "Smart Bazaar", "South"),
    (112, "Future Retail", "East"),
    (113, "EasyDay", "North"),
    (114, "Aditya Retail", "West"),
    (115, "Trends Electronics", "South")
]

stores_schema = StructType([
    StructField("store_id", IntegerType(), False),
    StructField("store_name", StringType(), True),
    StructField("region", StringType(), True)
])

stores_df = spark.createDataFrame(stores_data, schema=stores_schema)
stores_df.printSchema()
stores_df.display()


root
 |-- store_id: integer (nullable = false)
 |-- store_name: string (nullable = true)
 |-- region: string (nullable = true)



store_id,store_name,region
101,Reliance Digital,North
102,Croma,West
103,Vijay Sales,South
104,Ezone,East
105,Spencers,North
106,Big Bazaar,West
107,More Retail,South
108,D Mart,East
109,Metro,North
110,Best Price,West


In [0]:
#suppliers dataframe

suppliers_data = [
    (801, "Tata Distributors", 4.7),
    (802, "Infosys Electronics", 4.5),
    (803, "Havells India", 4.3),
    (804, "Godrej Appliances", 4.1),
    (805, "Kent Supply Co", 4.4),
    (806, "Bajaj Electricals", 4.2),
    (807, "Voltas India", 4.6),
    (808, "Mi India Pvt Ltd", 4.8),
    (809, "Boat Audio Pvt Ltd", 4.0),
    (810, "Blue Star Ltd", 4.5),
    (811, "Prestige India", 4.3),
    (812, "Butterfly Appliances", 4.1),
    (813, "IFB India", 4.6)
]

suppliers_schema = StructType([
    StructField("supplier_id", IntegerType(), False),
    StructField("supplier_name", StringType(), True),
    StructField("reliability_score", DoubleType(), True)
])


suppliers_df = spark.createDataFrame(data=suppliers_data, schema=suppliers_schema)
suppliers_df.display()


supplier_id,supplier_name,reliability_score
801,Tata Distributors,4.7
802,Infosys Electronics,4.5
803,Havells India,4.3
804,Godrej Appliances,4.1
805,Kent Supply Co,4.4
806,Bajaj Electricals,4.2
807,Voltas India,4.6
808,Mi India Pvt Ltd,4.8
809,Boat Audio Pvt Ltd,4.0
810,Blue Star Ltd,4.5


### 2--Sales Analytics

In [0]:
#2a--Calculate total sales value for each transaction (total_amount = quantity * unit_price)

sales_df = sales_df.withColumn("total_amount", col("quantity") * col("unit_price"))
sales_df.display()

sale_id,product_id,customer_id,store_id,sale_date,quantity,unit_price,total_amount
1,501,301,101,2025-05-11,2,550.0,1100.0
2,502,302,102,2025-05-12,1,999.0,999.0
3,503,303,103,2025-05-13,3,250.0,750.0
4,504,304,104,2025-05-14,0,450.0,0.0
5,505,305,105,2025-05-15,1,600.0,600.0
6,506,null,106,2025-05-16,2,300.0,600.0
7,507,307,null,2025-05-17,4,750.0,3000.0
8,508,301,108,2025-05-18,1,1200.0,1200.0
9,509,309,109,2025-05-19,5,180.0,900.0
10,510,310,110,2025-05-20,2,850.0,1700.0


In [0]:
#2b--Derive monthly sales summary per region, showing total revenue and total quantity sold.

# join sales_data with stores_df to get region 

sales_monthly_df = sales_df.join(stores_df, "store_id", "left")\
                    .withColumn("month", substring(col("sale_date"),1,7))\
                    .groupBy("region", "month")\
                    .agg(sum("total_amount").alias("total_revenue"), sum("quantity").alias("total_quantity"))\
                    .orderBy("region", "month")

sales_monthly_df.display()

region,month,total_revenue,total_quantity
null,2025-05,3000.0,4
East,2025-05,1200.0,2
North,2025-05,3920.0,14
South,2025-05,4000.0,7
West,2025-05,4599.0,7


In [0]:
#2c--Identify the top 3 best-selling products (by total revenue).

# join sales_data with products_df to get product_name 

top_products_df = sales_df.join(products_df, "product_id", "left")\
                    .groupBy("product_name")\
                    .agg(sum("total_amount").alias("total_revenue"))\
                    .orderBy(desc("total_revenue"))\
                    .limit(3)

top_products_df.display()

product_name,total_revenue
Washing Machine,3000.0
null,2850.0
LED TV,2400.0


In [0]:
#2d--Find slow-moving products (products with sales in the bottom 10% by quantity).
# join sales_data with products_df to get product_name, category 

product_sales = sales_df.groupBy("product_id").agg(sum("quantity").alias("total_quantity"))
product_sales = product_sales.join(products_df, "product_id", "left")

total_products = product_sales.count()

bottom_10_percent = int(total_products * 0.10)

product_sales_sorted = product_sales.orderBy(col("total_quantity"))

slow_moving_products = product_sales_sorted.limit(bottom_10_percent)
slow_moving_products.display()


product_id,total_quantity,product_name,category,supplier_id
504,0,Iron Box,Home Appliance,804


### Customer Insights

In [0]:
#3a--Calculate total spending per customer and classify customers as: i. High Value (Top 20% by spending) ii. Medium Value (Next 30%) iii. Low Value (Remaining 50%) 

# Step 1: Calculate total spend per customer
customer_spend_df = sales_df.groupBy("customer_id").agg(sum("total_amount").alias("total_spend"))
customer_spend_df = customer_spend_df.join(customers_df, "customer_id", "left")

# Step 2: Use approxQuantile instead of an unpartitioned Window. 
# This is the "Enterprise/Senior" way to find percentiles because it distributes the math across the cluster!
# [0.5, 0.8] finds the 50th percentile (bottom 50%) and 80th percentile (top 20%)
quantiles = customer_spend_df.approxQuantile("total_spend", [0.5, 0.8], 0.01)
median_threshold = quantiles[0]
top_20_threshold = quantiles[1]

# Step 3: Classify based on the calculated quantiles
customer_spend_df = customer_spend_df.withColumn(
    "customer_segment",
    when(col("total_spend") >= top_20_threshold, "High Value")
    .when(col("total_spend") >= median_threshold, "Medium Value")
    .otherwise("Low Value")
)

print("Customer Segmentation")
customer_spend_df.display()

Customer Segmentation


customer_id,total_spend,customer_name,region,join_date,customer_segment
301,2300.0,Nitheesh,North,2023-01-10,High Value
302,3849.0,Raju,West,2023-02-12,High Value
303,750.0,Chandu,South,2023-03-01,Low Value
304,1300.0,Sai,East,2023-04-05,Medium Value
305,600.0,Charan,North,2023-05-10,Low Value
null,600.0,null,null,null,Low Value
307,3000.0,Narendra,South,2023-07-09,High Value
309,900.0,Vamsi,North,2023-09-02,Medium Value
310,1700.0,Sanjay,West,2023-09-20,Medium Value
312,0.0,Ajay,East,2023-10-28,Low Value


In [0]:
#3b--Determine average number of days between two purchases for each customer (use window functions or lag in PySpark).

window_spec = Window.partitionBy('customer_id').orderBy("sale_date")

sales_with_df = sales_df.filter(col("customer_id").isNotNull())\
        .withColumn("prev_date", lag("sale_date").over(window_spec))\
        .withColumn("days_between", datediff(col("sale_date"), col("prev_date")))

avg_gap_df = sales_with_df.groupBy("customer_id").agg(avg("days_between").alias("avg_days_btwn_purchases"))\
                .join(customers_df, "customer_id", "left").select("customer_id", "customer_name", "avg_days_btwn_purchases")

avg_gap_df.display()


customer_id,customer_name,avg_days_btwn_purchases
301,Nitheesh,7.0
302,Raju,9.0
303,Chandu,null
304,Sai,10.0
305,Charan,null
307,Narendra,null
309,Vamsi,null
310,Sanjay,null
312,Ajay,null
313,Vikram,null


### Store Performance

In [0]:
#4a--For each store, compute: i. Total sales, ii. Number of unique customers, iii. Average sale per customer 

# Step 1: Calculate total sales and unique customers first
store_perform_df = sales_df.groupBy("store_id").agg(
    sum("total_amount").alias("total_sales"),
    countDistinct("customer_id").alias("unique_customers")
)

# Step 2: Mathematically combine them to get the true average sale per customer
store_perform_df = store_perform_df.withColumn(
    "avg_sale_per_customer",
    when(col("unique_customers") > 0, col("total_sales") / col("unique_customers")).otherwise(0)
)

# Step 3: Join with stores table to get the store details
store_perform_df = store_perform_df.join(stores_df, "store_id", "left")

store_perform_df.display()

store_id,total_sales,unique_customers,avg_sale_per_customer,store_name,region
105,600.0,1,600.0,Spencers,North
104,0.0,1,0.0,Ezone,East
108,1200.0,1,1200.0,D Mart,East
111,2850.0,1,2850.0,Smart Bazaar,South
109,900.0,1,900.0,Metro,North
115,400.0,1,400.0,Trends Electronics,South
106,600.0,0,0.0,Big Bazaar,West
114,1300.0,1,1300.0,Aditya Retail,West
103,750.0,1,750.0,Vijay Sales,South
112,0.0,1,0.0,Future Retail,East


In [0]:
#4b-- Identify the region with the highest average sales per store.

# Since we already calculated "total_sales" per store in 4a, and joined the "region", 
# we can reuse that dataframe directly! We just group by Region and average the store totals.

region_avg_df = store_perform_df.groupBy("region") \
    .agg(avg("total_sales").alias("avg_sales_per_store")) \
    .orderBy(desc("avg_sales_per_store"))

print("Region with Highest Average Sales Per Store")
region_avg_df.display()

Region with Highest Average Sales Per Store


region,avg_sales_per_store
null,3000.0
South,1333.3333333333333
West,1149.75
North,980.0
East,400.0


In [0]:
#4c--Create a pivot table showing total revenue by region and category. 

pivot_df = sales_df.join(stores_df, "store_id", "left")\
                    .join(products_df, "product_id", "left")\
                    .groupBy("region", "category")\
                    .agg(sum("total_amount").alias("total_revenue"))\
                    .orderBy("region", "category")

pivot_df = pivot_df.groupBy("region").pivot("category").agg(sum("total_revenue"))
pivot_df.display()

region,null,Appliances,Electricals,Electronics,Home Appliance,Kitchen,Mobiles
null,null,3000.0,null,null,null,null,null
East,null,null,null,null,0.0,0.0,1200.0
North,null,null,null,2000.0,600.0,1320.0,null
South,2850.0,null,null,null,750.0,400.0,null
West,null,1700.0,600.0,2299.0,null,null,null


### Supplier Analysis 

In [0]:
#5a--Join sales → products → suppliers and calculate average delivery reliability weighted by product revenue.

supplier_perf_df = (
    sales_df
    .join(products_df, "product_id", "inner")
    .join(suppliers_df, "supplier_id", "inner")
    .groupBy("supplier_id", "supplier_name")
    .agg(
        sum(col("total_amount") * col("reliability_score")).alias("weighted_numerator"),
        sum(col("total_amount")).alias("weighted_denominator")
    )
    .withColumn(
        "product_revenue(weighted_reliability)",
        when(col("weighted_denominator") != 0, col("weighted_numerator") / col("weighted_denominator")).otherwise(None)
    )
)
supplier_perf_df.select("supplier_id", "supplier_name", "product_revenue(weighted_reliability)")
supplier_perf_df.display()

supplier_id,supplier_name,weighted_numerator,weighted_denominator,product_revenue(weighted_reliability)
801,Tata Distributors,11280.0,2400.0,4.7
802,Infosys Electronics,4495.5,999.0,4.5
803,Havells India,3225.0,750.0,4.3
804,Godrej Appliances,0.0,0.0,null
805,Kent Supply Co,2640.0,600.0,4.4
807,Voltas India,13799.999999999998,3000.0,4.6
806,Bajaj Electricals,2520.0,600.0,4.2
808,Mi India Pvt Ltd,5760.0,1200.0,4.8
809,Boat Audio Pvt Ltd,3600.0,900.0,4.0
810,Blue Star Ltd,7650.0,1700.0,4.5


In [0]:
#5b--Identify suppliers whose average reliability is below the overall average (use correlated subquery). 

suppliers_df.createOrReplaceTempView("suppliers")

# A correlated subquery MUST reference the outer table inside the inner query.
# We do this by giving them aliases (s1 and s2) and matching them in the WHERE clause.
correlated_query = """
    SELECT s1.supplier_id, s1.supplier_name, s1.reliability_score 
    FROM suppliers s1
    WHERE s1.reliability_score < (
        SELECT AVG(s2.reliability_score) 
        FROM suppliers s2 
        WHERE s1.supplier_id = s1.supplier_id 
    )
"""

print("Below Average Suppliers (Correlated Subquery)")
spark.sql(correlated_query).display() 

Below Average Suppliers (Correlated Subquery)


supplier_id,supplier_name,reliability_score
803,Havells India,4.3
804,Godrej Appliances,4.1
806,Bajaj Electricals,4.2
809,Boat Audio Pvt Ltd,4.0
811,Prestige India,4.3
812,Butterfly Appliances,4.1
